# Cat vs. Dog Classifier (Transfer Learning)

Fine-tune a pretrained ResNet with very little data — standing on the shoulders of a model that already learned to see. **Runtime: GPU.**

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

## 2. Data: two CIFAR-10 classes (cat=3, dog=5)

In [ ]:
tfm = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
full = datasets.CIFAR10('.', train=True, download=True, transform=tfm)
idx = [i for i, (_, y) in enumerate(full) if y in (3, 5)][:2000]
dl = DataLoader(Subset(full, idx), batch_size=32, shuffle=True)
print(len(idx), 'cat/dog images')

## 3. Load a pretrained model and freeze it
Reuse the feature extractor; replace only the final layer.

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
for p in model.parameters():
    p.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)
print('trainable params:', sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4. Train only the head

In [ ]:
opt = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
remap = {3: 0, 5: 1}
model.train()
for xb, yb in dl:
    yb = torch.tensor([remap[int(y)] for y in yb]).to(device)
    xb = xb.to(device)
    opt.zero_grad()
    loss = nn.functional.cross_entropy(model(xb), yb)
    loss.backward()
    opt.step()
print('done, last loss', round(loss.item(), 3))

## Homework
1. Hold out a test split and report accuracy.
2. Unfreeze the last ResNet block and fine-tune gently (low LR).
3. Use your own two image folders with `ImageFolder`.
4. Compare to a small CNN trained from scratch on the same data.